In [ ]:
'''
Financial Sentiment Analysis: Tesla(TSLA) Case Study.
Author: [Gaurav Singh Parihar].
Objective: Correlate news sentiment with stock price using FinBERT.
'''

In [ ]:
!pip install yfinance transformers torch scipy

In [ ]:
import yfinance as yf

ticker = "TSLA"

df_prices = yf.download(ticker, period="1mo", interval="1d")

print(df_prices.head())

In [ ]:
import requests

api_key = 'e1ba4d9108da4e48925f4167f75d0063'
url = f'https://newsapi.org/v2/everything?q={ticker}&language=en&apiKey={api_key}'
response = requests.get(url)
news_data = response.json()

headlines = [article['title'] for article in news_data['articles']]
print(headlines[:5])

In [ ]:
from transformers import pipeline

sentiment_pipeline = pipeline("sentiment-analysis", model="ProsusAI/finbert")

results = sentiment_pipeline(headlines[:10])
for i, result in enumerate(results):
    print(f"Headline: {headlines[i]}")
    print(f"Sentiment: {result['label']}, Score: {result['score']}\n")

In [ ]:
# What: Running the headlines through the AI
# Why: To turn words into data we can graph later.
# How: We pass our list of headlines to the analyzer.
results = sentiment_pipeline(headlines)

# Let's create a nice table to see the results
import pandas as pd

# Create a new DataFrame for news
df_news = pd.DataFrame({
    'Headline': headlines,
    'Sentiment': [res['label'] for res in results],
    'Confidence': [res['score'] for res in results]
})

df_news.head(10) # Show the first 10 analyzed headlines

In [ ]:
df_news['Date'] = [a['publishedAt'][:10]for a in news_data['articles']]
df_news['Date'] = pd.to_datetime(df_news['Date'])

sentiment_map = {'positive': 1, 'neutral': 0, 'negative': -1}
df_news['Sentiment_Score'] = df_news['Sentiment'].map(sentiment_map)

df_news.head()

In [ ]:
daily_sentiment = df_news.groupby('Date')['Sentiment_Score'].mean().reset_index()

print("Daily Sentiment Summary:")
print(daily_sentiment.head())

In [ ]:
df_prices = df_prices.reset_index()
# Flatten the MultiIndex columns of df_prices
df_prices.columns = [col[0] if isinstance(col, tuple) else col for col in df_prices.columns]

# Remove timezone information from daily_sentiment['Date'] to match df_prices['Date']
daily_sentiment['Date'] = daily_sentiment['Date'].dt.tz_localize(None)

final_df = pd.merge(df_prices, daily_sentiment, on='Date', how='left')

final_df['Sentiment_Score'] = final_df['Sentiment_Score'].fillna(0)

final_df.head(10)

In [ ]:
import matplotlib.pyplot as plt

fig, ax1 = plt.subplots(figsize=(14, 7))

color = 'tab:red'
ax1.set_xlabel('Date')
ax1.set_ylabel('TSLA Close Price', color=color, fontsize=12)
ax1.plot(final_df['Date'], final_df['Close'], color=color, linewidth=2, label='Price')
ax1.tick_params(axis='y', labelcolor=color)

ax2 = ax1.twinx()
color = 'tab:red'
ax2.set_ylabel('Average Sentiment Score', color=color, fontsize=12)
ax2.axhline(0, color='black', linestyle='--', alpha=0.3)
ax2.bar(final_df['Date'], final_df['Sentiment_Score'], color=color, alpha=0.3, label='Sentiment')
ax2.tick_params(axis='y', labelcolor=color)

plt.title(f'TESLA: Price vs. News Sentiment (Last 30 Days)', fontsize = 16)
fig.tight_layout()
plt.show()

In [ ]:
correlation = final_df['Close'].corr(final_df['Sentiment_Score'])

print(f"The Correlation between Price and Sentiment is: {correlation:.2f}")

if correlation > 0.3:
  print("Insight: There is a positive relationship between news and price.")
else:
  print("Insight: News sentiment doe not strongly correlate with price in this window.")

In [ ]:
import seaborn as sns
import scipy.stats as stats

# 1. Create a figure with 3 subplots
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5))

# Plot A: Scatter Plot (Linearity Check)
sns.regplot(x='Sentiment_Score', y='Close', data=final_df, ax=ax1, color='purple')
ax1.set_title('Linearity: Price vs. Sentiment')

# Plot B: Price Distribution (Normality Check)
sns.histplot(final_df['Close'], kde=True, ax=ax2, color='blue')
ax2.set_title('Price Distribution')

# Plot C: Sentiment Distribution (Normality Check)
sns.histplot(final_df['Sentiment_Score'], kde=True, ax=ax3, color='red')
ax3.set_title('Sentiment Distribution')

plt.tight_layout()
plt.show()

In [ ]:
# We use scipy to get both the Correlation (r) and the P-value (p)
r, p_value = stats.pearsonr(final_df['Sentiment_Score'], final_df['Close'])

print(f"Pearson Correlation (r): {r:.4f}")
print(f"P-Value: {p_value:.4f}")

if p_value < 0.05:
    print("CONCLUSION: The correlation is statistically significant. We can trust this relationship.")
else:
    print("CONCLUSION: Not significant. We need more data (a longer time period) to be sure.")

In [ ]:
# What: Shifting the sentiment by 1 day
# Why: To see if news has a "Delayed" effect on the market.
final_df['Lagged_Sentiment'] = final_df['Sentiment_Score'].shift(1)

# Check the correlation again with the lag
lag_corr = final_df['Close'].corr(final_df['Lagged_Sentiment'])
print(f"Lagged Correlation: {lag_corr:.4f}")